In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/alifhafedz/housing_price/refs/heads/main/housing_price_dataset.csv')  # Replace with your file path
print("Initial Data Shape:", df.shape)
print(df.head())

Initial Data Shape: (50000, 6)
   SquareFeet  Bedrooms  Bathrooms Neighborhood  YearBuilt          Price
0        2126         4          1        Rural       1969  215355.283618
1        2459         3          2        Rural       1980  195014.221626
2        1860         2          1       Suburb       1970  306891.012076
3        2294         2          1        Urban       1996  206786.787153
4        2130         5          2       Suburb       2001  272436.239065


In [ ]:
for col in df.select_dtypes(include=np.number).columns:
    df[col].fillna(df[col].mean(), inplace=True)

# Fill missing categorical values with mode
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Remove duplicates
df.drop_duplicates(inplace=True)

# Remove outliers using Z-score for numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns
df = df[(np.abs(stats.zscore(df[numeric_cols])) < 3).all(axis=1)]

print("After Cleaning Shape:", df.shape)

After Cleaning Shape: (49965, 6)


/tmp/ipykernel_599/128284543.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mean(), inplace=True)
/tmp/ipykernel_599/128284543.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'd

In [ ]:
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# Encode categorical columns
for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

# Optional: Feature Engineering
# Example: create a 'TotalRooms' feature
if 'Bedrooms' in df.columns and 'Bathrooms' in df.columns:
    df['TotalRooms'] = df['Bedrooms'] + df['Bathrooms']

print("After Transformation Head:")
print(df.head())

After Transformation Head:
   SquareFeet  Bedrooms  Bathrooms  Neighborhood  YearBuilt     Price  \
0    0.207895  0.449282  -1.220097             0  -0.791876 -0.124972   
1    0.786793 -0.446574   0.005568             0  -0.260950 -0.392963   
2   -0.254528 -1.342431  -1.220097             1  -0.743610  1.080998   
3    0.499952 -1.342431  -1.220097             2   0.511307 -0.237861   
4    0.214849  1.345138   0.005568             1   0.752638  0.627062   

   TotalRooms  
0   -0.770815  
1   -0.441006  
2   -2.562528  
3   -2.562528  
4    1.350707  


In [ ]:
pca_cols = numeric_cols.tolist()
pca = PCA(n_components=min(5, len(pca_cols)))  # reduce to 5 principal components or fewer
principal_components = pca.fit_transform(df[pca_cols])

# Create a DataFrame with PCA components
pca_df = pd.DataFrame(principal_components, columns=[f'PC{i+1}' for i in range(principal_components.shape[1])])

# Combine PCA with any categorical features if needed
categorical_cols = df.select_dtypes(include='int64').columns.difference(numeric_cols)
final_df = pd.concat([pca_df, df[categorical_cols].reset_index(drop=True)], axis=1)

print("Final preprocessed Data Shape:", final_df.shape)
print(final_df.head())

Final preprocessed Data Shape: (49965, 6)
        PC1       PC2       PC3       PC4       PC5  Neighborhood
0  0.058670 -0.848250 -0.578726 -1.124368 -0.233025             0
1  0.246580 -0.457687 -0.042936  0.357725 -0.800579             0
2  0.468554 -1.923677  0.071855  0.072758  1.070757             1
3  0.065397 -1.466861  1.245726  0.003038 -0.388488             2
4  0.682484  1.158220  0.213002 -0.960605  0.199668             1


In [ ]:
final_df.to_csv('preprocessed_housing_data.csv', index=False)
print("Preprocessed data saved as 'preprocessed_housing_data.csv'")

Preprocessed data saved as 'preprocessed_housing_data.csv'


In [ ]:
!ls

housing_price  preprocessed_housing_data.csv  sample_data
